# SONICS — stratyfikowany podzbiór benchmarkowy (10%)

Ten notebook uruchamia skrypt tworzący deterministyczny, stratyfikowany podzbiór ~9.3k plików z pełnego zbioru SONICS (93k).

**Metodologia:**
- Stratyfikacja po `(set, spoof, label)` — 12 stratum
- Alokacja Hamilton (largest remainder) dla dokładnego 10% łącznie
- Seed `42` — pełna reprodukowalność
- Fizyczna kopia plików FLAC + `metadata.json` + manifest

**Ścieżki:**
- Metadane: `all_data/metadata.json`
- Audio: `all_data_16k_mono/`
- Wyjście: `benchmark_10pct/`

In [ ]:
from pathlib import Path
import subprocess
import sys

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve()
SONICS_BASE = Path("/net/people/plgrid/plgjedrzejkusnierz/scratch/data/Sonics")
METADATA = SONICS_BASE / "all_data" / "metadata.json"
AUDIO_DIR = SONICS_BASE / "all_data_16k_mono"
OUTPUT_DIR = SONICS_BASE / "benchmark_10pct"
SCRIPT = REPO_ROOT / "scripts" / "processing" / "create_sonics_benchmark_subset.py"

print(f"Script:   {SCRIPT}")
print(f"Metadata: {METADATA}")
print(f"Audio:    {AUDIO_DIR}")
print(f"Output:   {OUTPUT_DIR}")

## Uruchomienie skryptu

Ustaw `DRY_RUN = False`, aby skopiować pliki audio (~9.3k FLAC). Domyślnie `True` — tylko selekcja i raport.

In [ ]:
DRY_RUN = True  # zmień na False, aby wykonać fizyczną kopię plików

cmd = [
    sys.executable,
    str(SCRIPT),
    "--metadata", str(METADATA),
    "--audio-dir", str(AUDIO_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--fraction", "0.1",
    "--seed", "42",
    "--workers", "16",
]
if DRY_RUN:
    cmd.append("--dry-run")
else:
    cmd.append("--skip-existing")

result = subprocess.run(cmd, cwd=REPO_ROOT, check=False)
print(f"Exit code: {result.returncode}")
if result.returncode != 0:
    raise SystemExit("Subset creation failed — sprawdź raport powyżej.")

## Wizualizacja proporcji (full vs subset)

Porównanie rozkładu `set × spoof` między pełnym zbiorem a podzbiorem benchmarkowym.

In [ ]:
with open(METADATA, "r") as f:
    full_meta = pd.DataFrame(json.load(f))

manifest_path = OUTPUT_DIR / "all_data" / "selection_manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    subset_filenames = {item["filename"] for item in manifest["files"]}
    subset_meta = full_meta[full_meta["filename"].isin(subset_filenames)].copy()
    print(f"Loaded subset from manifest: {len(subset_meta):,} files")
else:
    import importlib.util

    spec = importlib.util.spec_from_file_location("subset_script", SCRIPT)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    subset_meta = mod.select_subset(full_meta, fraction=0.1, seed=42)
    print(f"Recomputed subset in-memory (dry-run): {len(subset_meta):,} files")


def split_spoof_table(df: pd.DataFrame, label: str) -> pd.Series:
    counts = df.groupby(["set", "spoof"]).size()
    counts.index = [f"{s}/{sp}" for s, sp in counts.index]
    counts.name = label
    return counts

full_counts = split_spoof_table(full_meta, "full")
subset_counts = split_spoof_table(subset_meta, "subset")
compare = pd.concat([full_counts, subset_counts], axis=1).fillna(0).astype(int)
compare["full_%"] = 100 * compare["full"] / compare["full"].sum()
compare["subset_%"] = 100 * compare["subset"] / compare["subset"].sum()
compare["delta_%"] = compare["subset_%"] - compare["full_%"]
display(compare)

splits = sorted(full_meta["set"].unique())
x = np.arange(len(splits))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
for spoof in ("bonafide", "deepfake"):
    full_vals = [full_meta[(full_meta["set"] == s) & (full_meta["spoof"] == spoof)].shape[0] for s in splits]
    sub_vals = [subset_meta[(subset_meta["set"] == s) & (subset_meta["spoof"] == spoof)].shape[0] for s in splits]
    offset = -width / 2 if spoof == "bonafide" else width / 2
    ax.bar(x + offset, full_vals, width, alpha=0.45, label=f"{spoof} (full)")
    ax.bar(x + offset, sub_vals, width, alpha=0.95, hatch="//", label=f"{spoof} (subset)")

ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel("Liczba próbek")
ax.set_title("Stratyfikacja set × spoof: pełny zbiór vs benchmark 10%")
ax.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()

## Raport sanity-check

Po pełnym uruchomieniu (bez `--dry-run`) raport jest zapisywany w `benchmark_10pct/selection_report.txt`.

In [ ]:
report_path = OUTPUT_DIR / "selection_report.txt"
if report_path.exists():
    print(report_path.read_text())
else:
    print("Brak zapisanego raportu (uruchom skrypt bez --dry-run).")